# Predicción de Churn — Telecom

Notebook de modelado: carga de features generadas en SQL, entrenamiento de modelos de clasificación y exportación de resultados para Power BI.

## Celda 1 — Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, recall_score)
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente')

## Celda 2 — Cargar datos

In [ ]:
# Ajusta la ruta si es necesario
df = pd.read_csv('../data/customers_features.csv')
print(f'Filas: {len(df)} | Columnas: {len(df.columns)}')
df.head(3)

## Celda 3 — Selección y codificación de variables

In [ ]:
# Variables para el modelo
features = [
    'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
    'score_servicios', 'ratio_cargo_antiguedad', 'cltv_proyectado_24m',
    'flag_contrato_mensual', 'flag_pago_riesgo',
    'gender', 'Partner', 'Dependents', 'InternetService',
    'PaperlessBilling', 'Contract', 'PaymentMethod',
    'segmento_antiguedad', 'segmento_valor'
]
X = df[features].copy()
y = df['Churn']

# Codificar variables categóricas automáticamente
cat_cols = X.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

print(f'Variables de entrada: {X.shape[1]}')
print(f'Tasa de churn (1=Sí): {y.mean():.2%}')

## Celda 4 — Dividir datos entrenamiento/prueba

In [ ]:
# 80% entrenamiento, 20% prueba. random_state fija la semilla para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar variables numéricas
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Entrenamiento: {len(X_train)} filas')
print(f'Prueba:        {len(X_test)} filas')

## Celda 5 — Modelo 1: Regresión Logística

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_proba_lr = lr.predict_proba(X_test_sc)[:, 1]

print('=== REGRESIÓN LOGÍSTICA ===')
print(classification_report(y_test, y_pred_lr, target_names=['No Churn','Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}')
print(f'Recall Churn: {recall_score(y_test, y_pred_lr):.4f}')

## Celda 6 — Modelo 2: Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    max_depth=8, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('=== RANDOM FOREST ===')
print(classification_report(y_test, y_pred_rf, target_names=['No Churn','Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'Recall Churn: {recall_score(y_test, y_pred_rf):.4f}')

## Celda 7 — Curva ROC comparativa

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, proba in [('Logistic Regression', y_proba_lr), ('Random Forest', y_proba_rf)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

ax.plot([0,1],[0,1], 'k--', label='Aleatorio (AUC=0.5)')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)')
ax.set_title('Curva ROC — Modelos de Churn')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/roc_curve.png', dpi=150)
plt.show()

## Celda 8 — Importancia de variables (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top10 = importances.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
top10.sort_values().plot(kind='barh', color='#2E86AB', ax=ax)
ax.set_title('Top 10 Variables más Importantes — Random Forest')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150)
plt.show()

print('Variables más importantes:')
for v, imp in top10.items():
    print(f'  {v:<35} {imp:.4f}')

## Celda 9 — Exportar predicciones para Power BI

In [ ]:
# Agregar probabilidades al dataset completo
X_all = df[features].copy()
for col in cat_cols:
    X_all[col] = le.fit_transform(X_all[col].astype(str))

df['prob_churn'] = rf.predict_proba(X_all)[:, 1]
df['pred_churn'] = rf.predict(X_all)

# Segmentar riesgo
def nivel_riesgo(p):
    if p >= 0.70: return 'Alto'
    elif p >= 0.40: return 'Medio'
    else: return 'Bajo'

df['nivel_riesgo'] = df['prob_churn'].apply(nivel_riesgo)

# Exportar
df.to_csv('../data/customers_scored.csv', index=False)
print('Archivo exportado: data/customers_scored.csv')
print(df['nivel_riesgo'].value_counts())